In [21]:
# from google.colab import drive
# drive.mount('/content/drive')

In [22]:
# ייבוא הספריות הדרושות
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.utils import resample
from sklearn.preprocessing import OneHotEncoder, LabelEncoder


In [31]:
# טעינת מערך הנתונים של Iris והפיכתו ל-DataFrame
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target

In [32]:
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [33]:
# הוספת ערכים חסרים באופן אקראי כדי להדגים טכניקות שונות
np.random.seed(42)
df.loc[df.sample(frac=0.1).index, 'sepal length (cm)'] = np.nan
df.loc[df.sample(frac=0.1).index, 'sepal width (cm)'] = np.nan

print("נתונים ראשוניים עם ערכים חסרים:")
print(df.head(10))



נתונים ראשוניים עם ערכים חסרים:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   
5                5.4               3.9                1.7               0.4   
6                4.6               3.4                1.4               0.3   
7                5.0               3.4                1.5               0.2   
8                4.4               2.9                1.4               0.2   
9                NaN               NaN                1.5               0.1   

   species  
0        0  
1        0  
2        0  
3        0  
4        0  
5        0  
6      

In [26]:
# אפשרויות לטיפול בנתונים חסרים

# 1. הורדת רשומות עם נתונים חסרים
df_dropped = df.dropna()
print("\nנתונים לאחר הסרת רשומות עם ערכים חסרים:")
print(df_dropped.head(10))

# 2. השלמת נתונים חסרים עם הערך הממוצע
imputer_mean = SimpleImputer(strategy='mean')
df['sepal length (cm)'] = imputer_mean.fit_transform(df[['sepal length (cm)']])

# 3. השלמת נתונים חסרים עם חציון (median)
imputer_median = SimpleImputer(strategy='median')
df['sepal width (cm)'] = imputer_median.fit_transform(df[['sepal width (cm)']])


נתונים לאחר הסרת רשומות עם ערכים חסרים:
    sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                 5.1               3.5                1.4               0.2   
1                 4.9               3.0                1.4               0.2   
2                 4.7               3.2                1.3               0.2   
3                 4.6               3.1                1.5               0.2   
4                 5.0               3.6                1.4               0.2   
5                 5.4               3.9                1.7               0.4   
6                 4.6               3.4                1.4               0.3   
7                 5.0               3.4                1.5               0.2   
8                 4.4               2.9                1.4               0.2   
10                5.4               3.7                1.5               0.2   

    species  
0         0  
1         0  
2         0  
3         0  
4       

In [27]:
# סקלות ונרמול
# לפני אימון המודל, יש צורך להבטיח שהתכונות יהיו בטווח ערכים דומה כדי שלא תהיינה תכונות שמשפיעות יותר מדי על המודל

# הדפסת הנתונים המקוריים
print("נתונים מקוריים:")
print(df[['sepal length (cm)', 'sepal width (cm)']].head(10))

# נרמול MinMax - מביא את הערכים לטווח שבין 0 ל-1
scaler_minmax = MinMaxScaler()
df[['sepal length (cm)_minmax', 'sepal width (cm)_minmax']] = scaler_minmax.fit_transform(df[['sepal length (cm)', 'sepal width (cm)']])

# הדפסת הנתונים לאחר נרמול MinMax
print("\nנתונים לאחר נרמול MinMax:")
print(df[['sepal length (cm)_minmax', 'sepal width (cm)_minmax']].head(10))

# נרמול Standard - מבצע סטנדרטיזציה כך שלכל תכונה יהיו ממוצע 0 וסטיית תקן 1
scaler_standard = StandardScaler()
df[['sepal length (cm)_standard', 'sepal width (cm)_standard']] = scaler_standard.fit_transform(df[['sepal length (cm)', 'sepal width (cm)']])

# הדפסת הנתונים לאחר נרמול Standard
print("\nנתונים לאחר נרמול Standard:")
print(df[['sepal length (cm)_standard', 'sepal width (cm)_standard']].head(10))


נתונים מקוריים:
   sepal length (cm)  sepal width (cm)
0           5.100000               3.5
1           4.900000               3.0
2           4.700000               3.2
3           4.600000               3.1
4           5.000000               3.6
5           5.400000               3.9
6           4.600000               3.4
7           5.000000               3.4
8           4.400000               2.9
9           5.833333               3.0

נתונים לאחר נרמול MinMax:
   sepal length (cm)_minmax  sepal width (cm)_minmax
0                  0.222222                 0.625000
1                  0.166667                 0.416667
2                  0.111111                 0.500000
3                  0.083333                 0.458333
4                  0.194444                 0.666667
5                  0.305556                 0.791667
6                  0.083333                 0.583333
7                  0.194444                 0.583333
8                  0.027778                 0.37500

In [28]:
# יצירת עמודה קטגוריאלית לדוגמה
df['category'] = ['red', 'blue', 'green', 'red', 'green', 'blue', 'red', 'green', 'blue', 'red'] * 15

# קידוד עם Label Encoding
# קידוד זה ממיר את הערכים הקטגוריאליים לערכים מספריים
label_encoder = LabelEncoder()
df['category_label_encoded'] = label_encoder.fit_transform(df['category'])

print("Label Encoding:")
print(df[['category', 'category_label_encoded']].head(10))

# קידוד עם One-Hot Encoding
# קידוד זה יוצר עמודה בינארית (0 או 1) עבור כל קטגוריה
one_hot_encoder = OneHotEncoder(sparse_output=False)
category_one_hot = one_hot_encoder.fit_transform(df[['category']])
category_one_hot_df = pd.DataFrame(category_one_hot, columns=one_hot_encoder.get_feature_names_out(['category']))

# איחוד של התוצאה עם DataFrame המקורי
df_one_hot_encoded = pd.concat([df, category_one_hot_df], axis=1)

print("\nOne-Hot Encoding:")
print(df_one_hot_encoded[['category'] + list(category_one_hot_df.columns)].head(10))


Label Encoding:
  category  category_label_encoded
0      red                       2
1     blue                       0
2    green                       1
3      red                       2
4    green                       1
5     blue                       0
6      red                       2
7    green                       1
8     blue                       0
9      red                       2

One-Hot Encoding:
  category  category_blue  category_green  category_red
0      red            0.0             0.0           1.0
1     blue            1.0             0.0           0.0
2    green            0.0             1.0           0.0
3      red            0.0             0.0           1.0
4    green            0.0             1.0           0.0
5     blue            1.0             0.0           0.0
6      red            0.0             0.0           1.0
7    green            0.0             1.0           0.0
8     blue            1.0             0.0           0.0
9      red          

In [29]:
# זיהוי והסרת ערכים חריגים עם IsolationForest
# בשלב זה כל הערכים החסרים הושלמו ולכן ניתן להשתמש ב-IsolationForest
iso_forest = IsolationForest(contamination=0.01, random_state=42)
outliers = iso_forest.fit_predict(df[['sepal length (cm)_standard', 'sepal width (cm)_standard']])

# הדפסת הערכים החריגים
print("ערכים חריגים (Outliers):")
print(df[outliers == -1])

# שמירת נתונים ללא הערכים החריגים
df_no_outliers = df[outliers == 1]
print("\nנתונים לאחר הסרת הערכים החריגים:")
print(df_no_outliers.head(10))


ערכים חריגים (Outliers):
     sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
15                 5.7               4.4                1.5               0.4   
131                7.9               3.8                6.4               2.0   

     species  sepal length (cm)_minmax  sepal width (cm)_minmax  \
15         0                  0.388889                     1.00   
131        2                  1.000000                     0.75   

     sepal length (cm)_standard  sepal width (cm)_standard category  \
15                    -0.169243                    3.35459     blue   
131                    2.623259                    1.87752     blue   

     category_label_encoded  
15                        0  
131                       0  

נתונים לאחר הסרת הערכים החריגים:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0           5.100000               3.5                1.4               0.2   
1           4.900000             

In [30]:
# איזון מחדש של הנתונים באמצעות דגימה חוזרת
# כאשר יש חוסר איזון בנתונים, המודל עלול להתמקד יותר בקבוצה הגדולה
# נבצע דגימה חוזרת (upsampling) כדי לאזן את הקבוצות
minority = df_no_outliers[df_no_outliers['species'] == 2]
majority = df_no_outliers[df_no_outliers['species'] != 2]
minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)
balanced_data = pd.concat([majority, minority_upsampled])

print("\nנתונים מאוזנים:")
print(balanced_data['species'].value_counts())


נתונים מאוזנים:
species
2    99
1    50
0    49
Name: count, dtype: int64
